# G03: ReAct + Calculator (Qwen3-4B), chuyên đề Mũ-Logarit, 3 dạng: D93, D52, D65

Notebook chạy **lần lượt 3 dạng** mà Qwen3-4B zero-shot chưa đạt 100%, nạp model **một lần**. Mỗi dạng chạy đủ
90 câu và ghi một file `<mã dạng>_react_calculator_full90.csv` trong `/kaggle/working`, cuối cùng ghi `summary_G03.csv`.

| Dạng | Nội dung | Zero-shot đúng | Số lệnh Calculator tối đa | Độ dài prompt (ký tự) |
|---|---|---|---|---|
| D93 | đếm số nguyên dương $a$ để có đúng ba số nguyên $b$ thỏa $(p^b-p)(a\cdot q^b-r)<0$ | 39/90 | 7 | 18,788 |
| D52 | đếm số nguyên $a$ để có ít nhất $k$ số nguyên $b\in(-L;L)$ thỏa $c^{a^2+b}\le d^{b-a}+e$ | 43/90 | 7 | 19,537 |
| D65 | tìm $P_{\min}$ của $P=x+y$ biết $\log_b\frac{c-xy}{px+qy}=b\,xy+px+qy-m$ | 47/90 | 9 | 20,715 |

Model **không tự tính tay bất kỳ phép nào**: mọi phép tính đều gọi tool `Calculator` (sympy, chính xác tuyệt đối, có bộ nhớ biến)
theo vòng lặp ReAct thật: `Thought → Action → Action Input → Observation → ...`

Prompt mỗi dạng gồm 5 phần giống bộ Số phức và dạng D79: `PART_TOOL` + `PART_KIENTHUC` (dùng chung) + `PART_HUONGGIAI`
(hướng giải theo lời giải mẫu của câu gốc) + `PART_FEWSHOT` (các `Observation` là kết quả chạy thật bằng chính backend) + `PART_NHIEMVU`.
Tham số sinh: `temperature=0.3, top_p=0.95, top_k=20, seed=42, presence_penalty=1.2, max_new_tokens=9216`, ép mở đầu `<think>\nThought:`.

## Trước khi chạy

1. Gắn Kaggle Dataset chứa `plan_solve_prompts_mulogarit_merged.json` (đã dùng cho zero-shot). Notebook tự dò đường dẫn bằng `glob`.
2. Bật GPU (khuyến nghị 2×T4), chọn **Save & Run All**.
3. Muốn thử nhanh vài câu: đặt `RUN_STT = [1, 2, 3]` và/hoặc `RUN_TYPES` ở ô cấu hình.


In [ ]:
!pip install -q -U vllm
!pip uninstall -y -q torchcodec torchaudio
import vllm; print('vLLM:', vllm.__version__)


In [ ]:
import os, re, json, time, glob
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

GROUP = 'G03'
RUN_TYPES = ['D93', 'D52', 'D65']      # the types of this notebook, in running order
DATA_FILENAME = 'plan_solve_prompts_mulogarit_merged.json'
# The dataset folder name under /kaggle/input differs between accounts, so
# search for the file instead of hardcoding a path.
_matches = glob.glob(f'/kaggle/input/**/{DATA_FILENAME}', recursive=True)
assert _matches, f'{DATA_FILENAME} not found under /kaggle/input. Add the dataset to the notebook first.'
DATA_PATH = _matches[0]

RUN_STT = None            # None = all 90 questions; for a quick test use e.g. [1, 2, 3]

MODEL = 'Qwen/Qwen3-4B'
# Same decoding settings as the Complex Numbers ReAct + Calculator runs.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.3, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_NEW_TOKENS = 9216

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(DATA_PATH, encoding='utf-8') as f:
    _all = json.load(f)
ALL_RECORDS = {ma: _all[ma] for ma in RUN_TYPES}
print({ma: len(v) for ma, v in ALL_RECORDS.items()})

tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)


In [ ]:
# =====================================================================
# PROMPTS. PART_TOOL and PART_KIENTHUC are shared by every problem type of the
# chapter; PART_HUONGGIAI, PART_FEWSHOT and PART_NHIEMVU are specific to each type.
# TYPE_PROMPTS[type]['template'] is the five parts joined; {de_bai} is the question.
# =====================================================================

PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: rate = log(6)/log(5)` → `Observation: rate = log(6)/log(5)`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: big = N(101**rate - 101, 20)`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Power is `**` (e.g. `x**2`, `101**rate`), not `^`. Square root is `sqrt(...)`; never the `√` symbol.
- Plain fractions are already exact: writing `7/2` gives exactly seven halves, never a decimal. You do NOT need any special function for fractions.
- Logarithms: `log(u)` is the NATURAL logarithm (write `log`, never `ln`). A logarithm to another base is written with the change of base formula, `log(u)/log(base)`, or equivalently `log(u, base)`. The exponential function is `exp(u)` and Euler's number is `E`.
- Absolute value is `Abs(u)`. Rounding: `floor(u)` is the greatest integer not above `u`, and `ceiling(u)` is the least integer not below `u`. `Min(u, v)` and `Max(u, v)` give the smaller and the larger of two numbers.
- Exact results stay exact (for example `log(8)/log(2)` becomes `3`). When you need a decimal value, for instance to see between which integers a number lies, write `N(expression, 20)`: this is the Calculator computing a decimal for you, not you approximating.
- To solve an equation, write `solve(expression, unknown)`, where the expression is the left side of the equation once everything is moved to one side so that it equals 0. It returns the list of solutions. A letter that the method tells you to use as an unknown (for example `x`, `y`, `t`, or the letters `a` and `b` of a formula) may be used inside `solve`, `diff`, `expand`, `factor` and `subs` without being assigned first; never store a number under that letter. Other tools: `expand(...)`, `factor(...)`, `diff(f, x)`, and `expr.subs(x, value)` to substitute.
- Some methods use these extra tools, always in exactly the form the method shows: `nsolve(expression, unknown, start)` finds a root numerically starting the search at `start`; `real_roots(polynomial)` lists the real roots of a polynomial exactly; `Sum(term, (k, first, last)).doit()` adds the terms exactly; `Count(condition, (n, low, high))` counts the integers `n` from `low` to `high` that make the condition true (it can be nested, and the counting letters need no earlier assignment); `Piecewise((value, condition), (other_value, True))` chooses a value by a condition; `And(...)` joins conditions; `oo` is infinity.
- To test whether two expressions are **exactly** equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`, decided exactly; never approximately. To compare two numbers by size, write `left < right`, `left <= right`, `left > right` or `left >= right`; the Calculator decides it with 50 digits of precision and answers `True` or `False`. Never write `==`.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Some equations genuinely have no simple closed form, such as a cubic, quartic, or higher degree polynomial that does not factor into nice roots: for those, `nroots(<polynomial>)` gives the Calculator's own numeric roots, exactly like a real handheld calculator's equation solving mode; this is still the Calculator computing, not you approximating. `nroots(...)` returns EVERY root of the polynomial, including non real ones; before doing anything else with a root, check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`) and discard it immediately if not. For a root confirmed real, use `re(root)`, not the raw value. Any comparison built on numeric roots should use a small tolerance instead of exact `Eq()`; everything that does not depend on a numeric root, in particular the final count and matching it against the answer options, stays exact as usual.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''

PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF EXPONENTIALS AND LOGARITHMS**

1. **Powers**: for $a>0$ and real $m,n$: $a^m a^n=a^{m+n}$, $\dfrac{a^m}{a^n}=a^{m-n}$, $(a^m)^n=a^{mn}$, $(ab)^n=a^n b^n$, $a^{-n}=\dfrac{1}{a^n}$, $a^0=1$, and $a^{m/n}=\sqrt[n]{a^m}$.
2. **Exponential function** $y=a^x$ with $a>0$, $a\ne1$: it is defined for every real $x$, and $a^x>0$ always (it never equals 0 and is never negative). It is strictly increasing when $a>1$ and strictly decreasing when $0<a<1$.
3. **Exponential equations**: $a^{f}=a^{g}\iff f=g$. For a number $c$: $a^{f}=c$ has a solution only when $c>0$, and then $f=\log_a c$.
4. **Exponential inequalities** (same base): if $a>1$, then $a^{f}<a^{g}\iff f<g$ (direction kept). If $0<a<1$, then $a^{f}<a^{g}\iff f>g$ (direction reversed). In particular the sign of $a^{u}-a^{v}$ equals the sign of $u-v$ when $a>1$.
5. **Substitution $t=a^x$**: because $a^x>0$, the new variable must satisfy $t>0$. Then $a^{2x}=t^2$ and $a^{x+k}=a^k\,t$.
6. **Definition of the logarithm**: for $a>0$, $a\ne1$, $b>0$: $\log_a b=c\iff a^c=b$. The base must be positive and not equal to 1, and the argument must be strictly positive.
7. **Basic values**: $\log_a 1=0$, $\log_a a=1$, $\log_a a^c=c$, and $a^{\log_a b}=b$ for $b>0$.
8. **Product, quotient, power** (for $x,y>0$): $\log_a(xy)=\log_a x+\log_a y$, $\log_a\dfrac{x}{y}=\log_a x-\log_a y$, $\log_a x^{n}=n\log_a x$. For an even power of a number $x\ne0$ that may be negative: $\log_a x^{2k}=2k\log_a|x|$. Also $\log_{a^m}x=\dfrac{1}{m}\log_a x$ and $\log_a\sqrt[n]{x}=\dfrac{1}{n}\log_a x$.
9. **Change of base**: $\log_a b=\dfrac{\log_c b}{\log_c a}$ for any base $c>0$, $c\ne1$. Consequences: $\log_a b=\dfrac{1}{\log_b a}$ (for $b\ne1$) and $\log_a b\cdot\log_b c=\log_a c$.
10. **Natural and common logarithm**: $\ln x=\log_e x$ and $\log x=\log_{10}x$. In particular $\log_a b=\dfrac{\ln b}{\ln a}$.
11. **Exchanging base and exponent**: $a^{\log_b c}=c^{\log_b a}$ for $a,c>0$ and $b>0$, $b\ne1$. Reason: taking $\log_b$ of each side gives $\log_b c\cdot\log_b a$ on both sides, and two positive numbers with the same logarithm are equal.
12. **Logarithmic function** $y=\log_a x$: its domain is $x>0$. It is strictly increasing when $a>1$ and strictly decreasing when $0<a<1$. If $a>1$: $\log_a u<\log_a v\iff 0<u<v$ (direction kept). If $0<a<1$: $\log_a u<\log_a v\iff u>v>0$ (direction reversed). In both cases the arguments must be positive first.
13. **Logarithmic equations**: first state the domain (every argument $>0$). Then $\log_a f=\log_a g\iff f=g$, and $\log_a f=c\iff f=a^c$. A solution is accepted only if it satisfies the domain.
14. **Logarithmic inequalities with a constant**: if $a>1$: $\log_a f>c\iff f>a^c$ and $\log_a f<c\iff 0<f<a^c$. If $0<a<1$: $\log_a f>c\iff 0<f<a^c$ and $\log_a f<c\iff f>a^c$.
15. **Substitution $t=\log_a x$**: here $x>0$ and $t$ can be any real number. Then $\log_a^2 x=t^2$ and $x=a^t$.
16. **Always find the domain first**: every logarithm argument $>0$, every base positive and not 1, every denominator $\ne0$, every expression under a square root $\ge0$. Any value found later is accepted only if it satisfies this domain.
17. **Sign of a difference of two logarithms** (base $a>1$, arguments $u,v>0$): the sign of $\log_a u-\log_a v$ equals the sign of $u-v$. In particular $\log_a u-\log_a v=0\iff u=v$.
18. **Products and square roots**: $A\cdot B=0$ means $A=0$ or $B=0$. For $A\cdot\sqrt{B}=0$: either $A=0$ (and then $B\ge0$ must hold so that the square root exists), or $B=0$. The sign of a product $A\cdot B$ is decided by the signs of its two factors: $A\cdot B\le 0$ means one factor is $\ge0$ and the other is $\le0$.
19. **Derivatives**: $(a^x)'=a^x\ln a$, $(e^x)'=e^x$, $(a^u)'=u'\,a^u\ln a$. Also $(\log_a x)'=\dfrac{1}{x\ln a}$, $(\ln x)'=\dfrac{1}{x}$, $(\log_a u)'=\dfrac{u'}{u\ln a}$, and for a quotient $\left(\dfrac{u}{v}\right)'=\dfrac{u'v-uv'}{v^2}$.
20. **Derivative and monotonicity**: if $f'(x)>0$ on an interval, $f$ is strictly increasing there; if $f'(x)<0$, it is strictly decreasing. A strictly increasing function satisfies $f(u)<f(v)\iff u<v$ and $f(u)=f(v)\iff u=v$, so the equation $f=k$ has at most one solution.
21. **A power function with exponent above 1**: if $\alpha>1$ then $f(t)=t^{\alpha}-t$ has $f'(t)=\alpha t^{\alpha-1}-1$, and for $t\ge1$ we get $t^{\alpha-1}\ge1$, so $f'(t)\ge\alpha-1>0$: $f$ is strictly increasing on $[1,+\infty)$.
22. **Comparing the bases**: if $a>b>1$ then $\log_b a=\dfrac{\ln a}{\ln b}>1$.
23. **Integers**: a sum or difference of integers is an integer, and a positive integer is at least 1. Different integer values of $t$ give different values of $y=t-x$ for a fixed integer $x$, so counting the integers $y$ is the same as counting the integers $t$.
24. **Counting integers**: for real numbers $p<q$, the number of integers strictly between them (open interval $(p,q)$) is $(\lceil q\rceil-1)-(\lfloor p\rfloor+1)+1=\lceil q\rceil-\lfloor p\rfloor-1$. The number of integers in the closed interval $[p,q]$ is $\lfloor q\rfloor-\lceil p\rceil+1$. Here $\lfloor u\rfloor$ is the greatest integer not above $u$ and $\lceil u\rceil$ is the least integer not below $u$. The integers $1,2,\dots,T$ are exactly $T$ numbers.
25. **Quadratic equation** $ax^2+bx+c=0$ ($a\ne0$): $\Delta=b^2-4ac$. If $\Delta>0$ there are two distinct real roots $x_{1,2}=\dfrac{-b\pm\sqrt{\Delta}}{2a}$; if $\Delta=0$ one double root; if $\Delta<0$ no real root. Viète: $x_1+x_2=-\dfrac{b}{a}$ and $x_1x_2=\dfrac{c}{a}$.
26. **Quadratic inequality**: for $a>0$ and $\Delta>0$ with roots $x_1<x_2$: $ax^2+bx+c<0\iff x_1<x<x_2$, and $ax^2+bx+c>0\iff x<x_1$ or $x>x_2$. The two roots themselves are included only for the non strict signs $\le$ and $\ge$.
27. **Two distinct positive roots**: the quadratic $at^2+bt+c=0$ has two distinct positive roots if and only if $\Delta>0$, $x_1+x_2>0$ and $x_1x_2>0$.
28. **Squares and square roots**: $\sqrt{u^2}=|u|$. For $k>0$: $x^2<k\iff-\sqrt{k}<x<\sqrt{k}$ and $x^2>k\iff x<-\sqrt{k}$ or $x>\sqrt{k}$.
29. **Inequality of the arithmetic and geometric means**: for $u,v\ge0$: $u+v\ge2\sqrt{uv}$, with equality if and only if $u=v$.
30. **Bunyakovsky inequality** (also called Cauchy and Schwarz): for real numbers $p,q,x,y$: $(px+qy)^2\le(p^2+q^2)(x^2+y^2)$, with equality if and only if $(p,q)$ and $(x,y)$ are proportional.
31. **Values of a continuous monotone function**: if $f$ is continuous and strictly increasing on $[p,q]$, then on the open interval $(p,q)$ it takes every value strictly between $f(p)$ and $f(q)$ exactly once, and no other value. So $f(x)=k$ has a solution in $(p,q)$ if and only if $f(p)<k<f(q)$. For a strictly decreasing $f$ the roles of $f(p)$ and $f(q)$ swap.
32. **Behaviour at the ends** (base $a>1$): $a^x\to0^+$ when $x\to-\infty$ and $a^x\to+\infty$ when $x\to+\infty$; $\ln x\to-\infty$ when $x\to0^+$ and $\ln x\to+\infty$ when $x\to+\infty$. A function that goes from $-\infty$ up to a local maximum, down to a local minimum, then up to $+\infty$ crosses the $x$ axis 3 times if the maximum is positive and the minimum is negative, and once otherwise.
33. **Quadratic in $t=a^x>0$**: the roots of $t^2+Bt+D=0$ are accepted only if they are positive. If $D<0$ the roots have opposite signs, and $t^2+Bt+D>0$ with $t>0$ means $t$ above the larger root. A negative sign in front of a number is part of the number: for example in $t^2-12t+35$ we have $B=-12$.
'''

TYPE_PROMPTS = {}

# ---------------------------------------------------------------- D93
D93_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE**

Problem shape: the problem gives $(p^b-p)(a\cdot q^b-r)<0$ with integers $p,q\ge2$ and $r\ge1$, and asks how many positive integers $a$ have exactly three integers $b$ satisfying it.

**The product is negative when its two factors have opposite signs.** The first factor $p^b-p$ is positive exactly when $b>1$, negative when $b<1$, and zero at $b=1$ (where the product is $0$, not negative). So we split into two cases.

**Case 1: $b>1$** (so $b\ge2$). The first factor is positive, so we need $a\cdot q^b-r<0$, that is $q^b<\dfrac ra$, that is $b<L$ with $L=\log_q\dfrac ra$. The integers $b$ are $2,3,\dots$ up to just below $L$. Exactly three of them means $b\in\{2,3,4\}$, which happens when $4<L\le5$, that is
$$q^4<\frac ra\le q^5\iff\frac{r}{q^5}\le a<\frac{r}{q^4}.$$
The positive integers $a$ in this range are $\left\lceil\frac r{q^5}\right\rceil,\dots,\left\lceil\frac r{q^4}\right\rceil-1$, and their number is $\left\lceil\frac r{q^4}\right\rceil-\left\lceil\frac r{q^5}\right\rceil$.

**Case 2: $b<1$** (so $b\le0$). The first factor is negative, so we need $a\cdot q^b-r>0$, that is $q^b>\dfrac ra$, that is $b>L$ with the same $L=\log_q\dfrac ra$. The integers $b$ are the integers above $L$ up to $0$. Exactly three of them means $b\in\{-2,-1,0\}$, which happens when $-3\le L<-2$, that is
$$q^{-3}\le\frac ra<q^{-2}\iff r\,q^2<a\le r\,q^3.$$
The integers $a$ in this range number $r\,q^3-r\,q^2$.

**The two cases cannot overlap.** Case 1 needs $L>4$ and Case 2 needs $L<-2$, so no $a$ belongs to both, and for an $a$ of one case the other case contributes no $b$. The total number of values of $a$ is the sum of the two counts. It is then compared with the four options.
'''

D93_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Có bao nhiêu số nguyên dương $a$ sao cho ứng với mỗi $a$ có đúng ba số nguyên $b$ thỏa mãn $(4^b-4)(a\cdot3^b-20)<0$?

A. $361$.
B. $359$.
C. $362$.
D. $360$.

<think>
Thought: I read the problem. The first factor is $4^b-4$; its base does not matter beyond the sign split. In the second factor $a\cdot3^b-20$ the base is $q=3$ and the number is $r=20$. In Case 1 the number of values of $a$ is $\lceil r/q^4\rceil-\lceil r/q^5\rceil$, with exact fractions. I store it as `case_one`.
Action: Calculator
Action Input: case_one = ceiling(20/3**4) - ceiling(20/3**5)
Observation: case_one = 0

Thought: In Case 2 the number of values of $a$ is $r\,q^3-r\,q^2$; with $q=3$ and $r=20$ it is an exact integer. I store it as `case_two`.
Action: Calculator
Action Input: case_two = 20*3**3 - 20*3**2
Observation: case_two = 360

Thought: The two cases never overlap, so the total is the sum.
Action: Calculator
Action Input: answer = case_one + case_two
Observation: answer = 360

Thought: The computation is done and its results are stored. Before testing anything, I copy all four options exactly as they are written in the problem, side by side: A: $361$, B: $359$, C: $362$, D: $360$. I will write each one in Calculator syntax, test them one call per option, and read each result next to the option I copied here.

Thought: Option A is $361$.
Action: Calculator
Action Input: test_A = Eq(answer, 361)
Observation: test_A = False

Thought: Option B is $359$.
Action: Calculator
Action Input: test_B = Eq(answer, 359)
Observation: test_B = False

Thought: Option C is $362$.
Action: Calculator
Action Input: test_C = Eq(answer, 362)
Observation: test_C = False

Thought: Option D is $360$.
Action: Calculator
Action Input: test_D = Eq(answer, 360)
Observation: test_D = True

Thought: Lining up what I copied with the stored results: A: $361$ gives False, B: $359$ gives False, C: $362$ gives False, D: $360$ gives True. Exactly one option matches, and it is D.

</think>
Final Answer: \boxed{D}
'''

D93_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] Read the problem and write down, without computing: the base $q$ and the number $r$ of the second factor $a\cdot q^b-r$ (the first factor $p^b-p$ is not needed for the numbers).
[Step 2] Store `case_one = ceiling(<r>/<q>**4) - ceiling(<r>/<q>**5)`, writing the numbers from the problem.
[Step 3] Store `case_two = <r>*<q>**3 - <r>*<q>**2`.
[Step 4] Store `answer = case_one + case_two`.
[Step 5] Copy all four options side by side in one Thought, and write each one in Calculator syntax (a fraction `\dfrac{p}{q}` becomes `p/q`, `\sqrt{n}` becomes `sqrt(n)`, a power `u^{k}` becomes `u**k`). Then test every option in order with `test_A = Eq(answer, <option A>)`, then `test_B`, `test_C`, `test_D` in the same way. Finally line up each copied option next to its stored result; exactly one of them is `True`. Do not pick the letter by eye.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''

TYPE_PROMPTS['D93'] = {
    'template': PART_TOOL + PART_KIENTHUC + D93_HUONGGIAI + D93_FEWSHOT + D93_NHIEMVU,
    'free_unknowns': [],
    'max_tool_calls': 25,
}


# ---------------------------------------------------------------- D52
D52_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE**

Problem shape: the problem gives the inequality $c^{a^2+b}\le d^{\,b-a}+e$ (with integers $c,d\ge2$ and $e>0$), an open window $(-L;L)$ for $b$ and a number $k$, and asks how many integers $a$ have at least $k$ integers $b$ in the window satisfying the inequality.

**Looking at the inequality as a condition on $b$.** Divide by $c^{b}>0$:
$$c^{a^2}\le d^{-a}\left(\frac dc\right)^{b}+e\,c^{-b}.$$
When $c>d$, both terms on the right decrease as $b$ grows (the base $\frac dc<1$ and the base $\frac1c<1$), so the right side is a decreasing function of $b$ and the integers $b$ that satisfy the inequality form a prefix of the window, $b\le\alpha$. Then "at least $k$ integers" simply means that the first $k$ integers of the window, $-L+1,\dots,-L+k$, all satisfy it, which is decided by testing the single value $b=-L+k$. When $c<d$ the term $\left(\frac dc\right)^b$ grows with $b$, and the satisfying integers do not have to be a prefix of the window. A safe way that works in every case is to test **every integer $b$ of the window** for each $a$.

**Which $a$ matter.** The left side $c^{a^2+b}$ grows extremely fast with $|a|$, much faster than the right side $d^{b-a}+e$. For $|a|>10$ no integer $b$ of the window can satisfy the inequality, so it is enough to examine the integers $a$ from $-10$ to $10$ (this is the table of values used in the original solution).

**Counting.** The window contains the integers $b=-L+1,\dots,L-1$. For a fixed $a$, the number of window integers $b$ that satisfy $c^{a^2+b}\le d^{b-a}+e$ is counted; the value $a$ is accepted when this number is at least $k$. The answer is the number of accepted integers $a$ in $[-10,10]$.

**The Calculator can do this table for you.** The function `Count(condition, (n, low, high))` returns how many integers $n$ from `low` to `high` make the condition true, deciding the condition exactly for each integer. It can be nested: the inner `Count` runs over $b$ for each value of $a$, and the outer `Count` runs over $a$ and keeps the values for which the inner count is at least $k$. The answer is then compared with the four options.
'''

D52_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Có bao nhiêu số nguyên $a$ sao cho có ít nhất 3 số nguyên $b\in(-7;7)$ thỏa mãn $2^{a^2+b}\le 3^{b-a}+8$?

A. $4$.
B. $6$.
C. $5$.
D. $3$.

<think>
Thought: I read the problem. The window is $(-7;7)$, so $L=7$. The open window starts after $-7$, so its first integer is $-7+1$; I store it as `first_b`.
Action: Calculator
Action Input: first_b = -7 + 1
Observation: first_b = -6

Thought: The open window ends before $7$, so its last integer is $7-1$; I store it as `last_b`.
Action: Calculator
Action Input: last_b = 7 - 1
Observation: last_b = 6

Thought: Now the table. The left base is 2, the right base is 3, the added constant is 8 and the required number of integers $b$ is 3. The inner Count runs over $b$ from `first_b` to `last_b` and counts how many satisfy $2^{a^2+b}\le3^{b-a}+8$; the outer Count runs over $a$ from $-10$ to $10$ and keeps the values of $a$ for which that inner count is at least 3. The result is the number of integers $a$.
Action: Calculator
Action Input: answer = Count(Count(2**(a**2 + b) <= 3**(b - a) + 8, (b, first_b, last_b)) >= 3, (a, -10, 10))
Observation: answer = 5

Thought: The computation is done and its results are stored. Before testing anything, I copy all four options exactly as they are written in the problem, side by side: A: $4$, B: $6$, C: $5$, D: $3$. I will write each one in Calculator syntax, test them one call per option, and read each result next to the option I copied here.

Thought: Option A is $4$.
Action: Calculator
Action Input: test_A = Eq(answer, 4)
Observation: test_A = False

Thought: Option B is $6$.
Action: Calculator
Action Input: test_B = Eq(answer, 6)
Observation: test_B = False

Thought: Option C is $5$.
Action: Calculator
Action Input: test_C = Eq(answer, 5)
Observation: test_C = True

Thought: Option D is $3$.
Action: Calculator
Action Input: test_D = Eq(answer, 3)
Observation: test_D = False

Thought: Lining up what I copied with the stored results: A: $4$ gives False, B: $6$ gives False, C: $5$ gives True, D: $3$ gives False. Exactly one option matches, and it is C.

</think>
Final Answer: \boxed{C}
'''

D52_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] Read the problem and write down, without computing: the base $c$ on the left $c^{a^2+b}$; the base $d$ and the added constant $e$ on the right $d^{b-a}+e$; the number $k$ in "at least $k$ integers $b$"; and the number $L$ of the window $(-L;L)$.
[Step 2] Store `first_b = -<L> + 1`, then `last_b = <L> - 1`, writing the number $L$ from the problem.
[Step 3] Store `answer = Count(Count(<c>**(a**2 + b) <= <d>**(b - a) + <e>, (b, first_b, last_b)) >= <k>, (a, -10, 10))`, writing the numbers from the problem. The letters `a` and `b` here are the summation variables of the Count and need no earlier assignment.
[Step 4] Copy all four options side by side in one Thought, and write each one in Calculator syntax (a fraction `\dfrac{p}{q}` becomes `p/q`, `\sqrt{n}` becomes `sqrt(n)`, a power `u^{k}` becomes `u**k`). Then test every option in order with `test_A = Eq(answer, <option A>)`, then `test_B`, `test_C`, `test_D` in the same way. Finally line up each copied option next to its stored result; exactly one of them is `True`. Do not pick the letter by eye.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''

TYPE_PROMPTS['D52'] = {
    'template': PART_TOOL + PART_KIENTHUC + D52_HUONGGIAI + D52_FEWSHOT + D52_NHIEMVU,
    'free_unknowns': [],
    'max_tool_calls': 25,
}


# ---------------------------------------------------------------- D65
D65_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE**

Problem shape: positive reals $x,y$ satisfy $\log_b\dfrac{c-xy}{px+qy}=b\,xy+px+qy-m$ (with $m=bc+1$), and we want the minimum of $P=x+y$.

**Turning the equation into one relation.** Split the logarithm: $\log_b(c-xy)-\log_b(px+qy)=b\,xy+px+qy-m$. Since $m=bc+1$, we have $b\,xy-m=-b(c-xy)-1$, and $-1=-\log_b b$, so the equation can be written
$$\log_b\big[b(c-xy)\big]+b(c-xy)=\log_b(px+qy)+(px+qy).$$
Let $f(t)=\log_b t+t$ for $t>0$. Its derivative $f'(t)=\dfrac{1}{t\ln b}+1>0$, so $f$ is strictly increasing. The equation says $f\big(b(c-xy)\big)=f(px+qy)$, hence
$$b(c-xy)=px+qy\iff y=\frac{bc-px}{bx+q}.$$
Because $y>0$, we need $0<x<\dfrac{bc}{p}$.

**Minimising.** Now $P(x)=x+\dfrac{bc-px}{bx+q}$ on $\left(0;\dfrac{bc}{p}\right)$. Its derivative is
$$P'(x)=1-\frac{pq+b^2c}{(bx+q)^2}.$$
Set $w=\sqrt{pq+b^2c}$. Then $P'(x)=0$ when $bx+q=w$ (the other sign is negative and rejected), that is $x^*=\dfrac{w-q}{b}$. The derivative is negative before $x^*$ and positive after it, so $P$ has its minimum there, provided $x^*$ lies in $\left(0;\dfrac{bc}{p}\right)$.

**The minimum value.** At $x^*$ we have $bx^*+q=w$ and $y=\dfrac{bc-px^*}{w}$. Putting these together and using $w^2=pq+b^2c$, the value simplifies to
$$P_{\min}=\frac{2w-(p+q)}{b}.$$
Check on the original problem ($b=3$, $c=1$, $p=1$, $q=2$): $w=\sqrt{11}$ and $P_{\min}=\dfrac{2\sqrt{11}-3}{3}$.

**Conclusion.** We compute $w$, confirm that $x^*$ lies in the allowed interval, and compute $P_{\min}$ exactly. It is then compared with the four options; an option may be written in a different but equal form (for example $-\frac{15}{4}+\frac{\sqrt{2145}}{4}$ instead of $\frac{2\sqrt{2145}-30}{8}$), so each option is tested by its numerical value.
'''

D65_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Xét các số thực dương $x,y$ thỏa mãn $\log_{5}\dfrac{7-xy}{3x+4y}=5xy+3x+4y-36$. Tìm giá trị nhỏ nhất $P_{\min}$ của $P=x+y$.

A. $P_{\min}=\dfrac{7}{5} + \dfrac{2 \sqrt{187}}{5}$.
B. $P_{\min}=- \dfrac{7}{5} + \dfrac{2 \sqrt{187}}{5}$.
C. $P_{\min}=- \dfrac{7}{5} + \dfrac{\sqrt{187}}{5}$.
D. $P_{\min}=- \dfrac{7}{10} + \dfrac{\sqrt{187}}{5}$.

<think>
Thought: I read the problem. The base is $b=5$, the constant in the numerator is $c=7$, the coefficients in the denominator are $p=3$ (of $x$) and $q=4$ (of $y$). The method uses $w=\sqrt{pq+b^2c}$; I store it as `critical_root`, exact, with no decimals.
Action: Calculator
Action Input: critical_root = sqrt(3*4 + 5**2*7)
Observation: critical_root = sqrt(187)

Thought: The critical point is $x^*=\dfrac{w-q}{b}$. I store it as `x_star`.
Action: Calculator
Action Input: x_star = (critical_root - 4)/5
Observation: x_star = -4/5 + sqrt(187)/5

Thought: The allowed interval for $x$ is $(0;\,bc/p)$. I store its right end $bc/p$ as `x_limit`.
Action: Calculator
Action Input: x_limit = 5*7/3
Observation: x_limit = 35/3

Thought: The critical point must lie inside the interval: its distance to 0 and its distance to `x_limit` must both be positive, so I ask the Calculator whether the smaller of `x_star` and `x_limit - x_star` is above 0.
Action: Calculator
Action Input: domain_check = Min(x_star, x_limit - x_star) > 0
Observation: domain_check = True

Thought: The minimum value is $P_{\min}=\dfrac{2w-(p+q)}{b}$. I store it as `answer`.
Action: Calculator
Action Input: answer = (2*critical_root - (3 + 4))/5
Observation: answer = -7/5 + 2*sqrt(187)/5

Thought: The computation is done and its results are stored. Before testing anything, I copy all four options exactly as they are written in the problem, side by side: A: $P_{\min}=\dfrac{7}{5} + \dfrac{2 \sqrt{187}}{5}$, B: $P_{\min}=- \dfrac{7}{5} + \dfrac{2 \sqrt{187}}{5}$, C: $P_{\min}=- \dfrac{7}{5} + \dfrac{\sqrt{187}}{5}$, D: $P_{\min}=- \dfrac{7}{10} + \dfrac{\sqrt{187}}{5}$. I will write each one in Calculator syntax, test them one call per option, and read each result next to the option I copied here.

Thought: Option A is $P_{\min}=\dfrac{7}{5} + \dfrac{2 \sqrt{187}}{5}$; in Calculator syntax this is `7/5+(2*sqrt(187))/5`.
Action: Calculator
Action Input: test_A = Abs(answer - (7/5+(2*sqrt(187))/5)) < 10**(-9)
Observation: test_A = False

Thought: Option B is $P_{\min}=- \dfrac{7}{5} + \dfrac{2 \sqrt{187}}{5}$; in Calculator syntax this is `-7/5+(2*sqrt(187))/5`.
Action: Calculator
Action Input: test_B = Abs(answer - (-7/5+(2*sqrt(187))/5)) < 10**(-9)
Observation: test_B = True

Thought: Option C is $P_{\min}=- \dfrac{7}{5} + \dfrac{\sqrt{187}}{5}$; in Calculator syntax this is `-7/5+sqrt(187)/5`.
Action: Calculator
Action Input: test_C = Abs(answer - (-7/5+sqrt(187)/5)) < 10**(-9)
Observation: test_C = False

Thought: Option D is $P_{\min}=- \dfrac{7}{10} + \dfrac{\sqrt{187}}{5}$; in Calculator syntax this is `-7/10+sqrt(187)/5`.
Action: Calculator
Action Input: test_D = Abs(answer - (-7/10+sqrt(187)/5)) < 10**(-9)
Observation: test_D = False

Thought: Lining up what I copied with the stored results: A: $P_{\min}=\dfrac{7}{5} + \dfrac{2 \sqrt{187}}{5}$ gives False, B: $P_{\min}=- \dfrac{7}{5} + \dfrac{2 \sqrt{187}}{5}$ gives True, C: $P_{\min}=- \dfrac{7}{5} + \dfrac{\sqrt{187}}{5}$ gives False, D: $P_{\min}=- \dfrac{7}{10} + \dfrac{\sqrt{187}}{5}$ gives False. Exactly one option matches, and it is B.

</think>
Final Answer: \boxed{B}
'''

D65_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] Read the problem and write down, without computing: the base $b$ of the logarithm; the constant $c$ in the numerator $c-xy$; the coefficient $p$ of $x$ and the coefficient $q$ of $y$ in the denominator $px+qy$ (a coefficient that is not written is 1).
[Step 2] Store `critical_root = sqrt(<p>*<q> + <b>**2*<c>)`, writing the numbers from the problem. This is $w=\sqrt{pq+b^2c}$, kept exact.
[Step 3] Store `x_star = (critical_root - <q>)/<b>`, then `x_limit = <b>*<c>/<p>`.
[Step 4] Store `domain_check = Min(x_star, x_limit - x_star) > 0`. It must be `True`.
[Step 5] Store `answer = (2*critical_root - (<p> + <q>))/<b>`.
[Step 6] Copy all four options side by side in one Thought, and write each one in Calculator syntax (a fraction `\dfrac{p}{q}` becomes `p/q`, `\sqrt{n}` becomes `sqrt(n)`, a power `u^{k}` becomes `u**k`). The options are written as sums of fractions and square roots; write each one as a single expression, for example `-7/5 + 2*sqrt(187)/5`. Then test every option in order with `test_A = Abs(answer - (<option A>)) < 10**(-9)`, then `test_B`, `test_C`, `test_D` in the same way. Finally line up each copied option next to its stored result; exactly one of them is `True`. Do not pick the letter by eye.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''

TYPE_PROMPTS['D65'] = {
    'template': PART_TOOL + PART_KIENTHUC + D65_HUONGGIAI + D65_FEWSHOT + D65_NHIEMVU,
    'free_unknowns': [],
    'max_tool_calls': 25,
}


In [ ]:
# The context window must hold the longest prompt plus the whole generation budget.
PROMPT_TOKENS = 0
for ma, spec in TYPE_PROMPTS.items():
    longest = max(ALL_RECORDS[ma], key=lambda r: len(r['de_bai_mcq']))
    n_tok = len(tok(spec['template'].replace('{de_bai}', longest['de_bai_mcq'])).input_ids)
    print(f'{ma}: prompt {n_tok} tokens')
    PROMPT_TOKENS = max(PROMPT_TOKENS, n_tok)
MAX_MODEL_LEN = ((PROMPT_TOKENS + MAX_NEW_TOKENS + 1500) // 1024 + 1) * 1024
print(f'Longest prompt: {PROMPT_TOKENS} tokens -> MAX_MODEL_LEN={MAX_MODEL_LEN}')

N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'GPUs: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype='float16',
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90,
          trust_remote_code=True, enforce_eager=True, seed=SEED)
print('Ready.')


In [ ]:
# =====================================================================
# BACKEND: the Calculator tool (SymPy with variable memory)
# =====================================================================
import re

import sympy as sp

ASSIGN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)
# Names that already mean something to the Calculator (functions such as N, Min, floor and
# constants such as E, I, S). Storing a value under one of them would silently replace it,
# for example "N = 94" would make the later call N(...) fail for the rest of the question.
RESERVED_NAMES = {n for n in dir(sp) if not n.startswith('_')}
RELATIONAL = (sp.StrictLessThan, sp.LessThan, sp.StrictGreaterThan, sp.GreaterThan, sp.Unequality)


class Count(sp.Expr):
    """Count(condition, (n, low, high)): how many integers n from low to high make the condition true.

    The condition is decided exactly, one integer at a time (like a calculator table). It may itself
    contain another Count over a second variable; the inner Count is evaluated only after the outer
    variable has been replaced by a number, so nothing is ever expanded symbolically."""

    def __new__(cls, condition, limits, low=None, high=None):
        if low is None:
            var, low, high = limits
        else:
            var = limits
        obj = sp.Expr.__new__(cls, condition, var, sp.sympify(low), sp.sympify(high))
        if getattr(condition, 'free_symbols', set()) - {var}:
            return obj                      # still depends on another variable: wait
        return obj.doit()

    @property
    def free_symbols(self):
        condition, var, low, high = self.args
        return (condition.free_symbols - {var}) | low.free_symbols | high.free_symbols

    def doit(self, **hints):
        condition, var, low, high = self.args
        total = 0
        for value in range(int(low), int(high) + 1):
            decided = condition.subs(var, sp.Integer(value))
            decided = decided.doit() if hasattr(decided, 'doit') else decided
            if decided is sp.true or decided is True:
                total += 1
            elif not (decided is sp.false or decided is False):
                raise ValueError(f'the condition could not be decided for {var} = {value}: {decided}')
        return sp.Integer(total)


EXTRA_NAMES = {'Count': Count}
RESERVED_NAMES = RESERVED_NAMES | set(EXTRA_NAMES)


class Calculator:
    """SymPy calculator with variable memory (one fresh instance per question)."""

    def __init__(self, free_unknowns=()):
        self.free_unknowns = set(free_unknowns)
        # Free unknowns are registered up front as REAL symbols; letting
        # sympify create them lazily would make them generic complex symbols.
        self.ns = {name: sp.Symbol(name, real=True) for name in self.free_unknowns}

    def _undefined_names(self, expr):
        free = getattr(expr, 'free_symbols', set())
        missing = {str(s) for s in free} - self.free_unknowns
        if not missing:
            return None
        unknown = ', '.join(sorted(missing))
        in_memory = ', '.join(sorted(self.ns)) or '(nothing stored yet)'
        allowed = ', '.join(sorted(self.free_unknowns)) or '(none)'
        return (f'undefined name(s): {unknown}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {in_memory}. Declared free unknowns '
                f'(allowed without prior assignment): {allowed}. Either compute '
                'and store that name first (with "name = expression"), or '
                'rewrite the expression without it.')

    def _unknown_functions(self, parsed):
        """Names used as functions that the Calculator does not know (ceil, log10, int, ...).

        SymPy accepts any unknown name followed by parentheses and silently keeps it as an
        unevaluated symbol, so "count = ceil(115.36) - 1" would be stored as if it were a
        result. Refuse it with a message that says what to write instead."""
        items = parsed if isinstance(parsed, (list, tuple, sp.FiniteSet)) else [parsed]
        names = set()
        for item in items:
            for node in getattr(item, 'atoms', lambda *a: set())(sp.core.function.AppliedUndef):
                names.add(type(node).__name__)
        if not names:
            return None
        return (f'unknown function(s): {", ".join(sorted(names))}. The Calculator does not know '
                'these names, so nothing was computed. Use only: log, exp, sqrt, floor, '
                'ceiling, Abs, Min, Max, N, solve, expand, factor, diff, nroots, Eq. '
                'For example: round up with ceiling(u), round down with floor(u), a logarithm '
                'to base b as log(u)/log(b).')

    def _decide_relational(self, rel):
        """True or False for an inequality between constant expressions."""
        err = self._undefined_names(rel.lhs - rel.rhs)
        if err:
            return None, err
        if rel.lhs.free_symbols or rel.rhs.free_symbols:
            return None, ('cannot decide an inequality that still contains an '
                          'unknown. Substitute numbers first.')
        diff = sp.N(rel.lhs - rel.rhs, 50)
        if abs(diff) < sp.Float('1e-40'):
            diff = sp.Integer(0)
        table = {sp.StrictLessThan: diff < 0, sp.LessThan: diff <= 0,
                 sp.StrictGreaterThan: diff > 0, sp.GreaterThan: diff >= 0,
                 sp.Unequality: diff != 0}
        return ('True' if table[type(rel)] else 'False'), None

    def _boolean_result(self, name, truth):
        """Format a True/False answer; store it when it was assigned to a name."""
        text = 'True' if truth else 'False'
        if name:
            self.ns[name] = sp.true if truth else sp.false
            return f'{name} = {text}', None
        return text, None

    def evaluate(self, expression):
        """Return (result_text, error_text). Exactly one of them is None."""
        s = expression.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        name = None
        m = ASSIGN_RE.match(s)
        # "x = Eq(...)" is an assignment; a bare "Eq(a, b)" has no top level "=".
        if m and not s.lstrip().startswith('Eq('):
            name, s = m.group(1), m.group(2)
            if name in self.free_unknowns:
                return None, (f'"{name}" is a declared unknown used by solve(...) and cannot be '
                              'a variable name. Choose another name and store the value again.')
            if name in RESERVED_NAMES:
                return None, (f'"{name}" cannot be used as a variable name because it is the name '
                              'of a Calculator function or constant (for example N(expression, '
                              'digits) computes a decimal value). Choose another name, such as a '
                              'descriptive word, and store the value again.')

        try:
            parsed = sp.sympify(s, locals={**EXTRA_NAMES, **self.ns})
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        err = self._unknown_functions(parsed)
        if err:
            return None, err

        try:
            if isinstance(parsed, sp.Equality):
                err = self._undefined_names(parsed.lhs - parsed.rhs)
                if err:
                    return None, err
                same = sp.simplify(parsed.lhs - parsed.rhs) == 0
                return self._boolean_result(name, same)

            if isinstance(parsed, RELATIONAL):
                text, err = self._decide_relational(parsed)
                if err:
                    return None, err
                return self._boolean_result(name, text == 'True')

            if isinstance(parsed, (sp.logic.boolalg.BooleanTrue, sp.logic.boolalg.BooleanFalse)):
                return self._boolean_result(name, bool(parsed))

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Result of solve(...): a list of solutions. Check each element
                # separately instead of simplifying the list as a whole.
                items = list(parsed)
                for item in items:
                    err = self._undefined_names(item)
                    if err:
                        return None, err
                if name:
                    self.ns[name] = items
                    return f'{name} = {items}', None
                return str(items), None

            value = sp.expand(sp.simplify(parsed))
            err = self._undefined_names(value)
            if err:
                return None, err
            if name:
                self.ns[name] = value
                return f'{name} = {value}', None
            return str(value), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'


# =====================================================================
# BATCHED ReAct LOOP (one function call = one problem type, all its questions):
# one vLLM call per round for every active question, then the Calculator
# runs separately for each question.
# =====================================================================
STOP_STR = 'Observation:'
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')


class QuestionState:
    """ReAct state of ONE question (own variable memory, own token budget)."""

    def __init__(self, rec, chat_prompt, free_unknowns):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.calculator = Calculator(free_unknowns=free_unknowns)
        # Forced prefix: the model must continue from "Thought:" right after
        # <think>, it cannot start with free prose.
        self.full_text = '<think>\nThought:'
        self.total_tokens = 0
        self.n_calls = 0
        self.tool_log = []
        self.done = False
        self.stop_reason = ''
        self.final_round = False          # tool limit reached: force the Final Answer


def extract_letter(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*(?:\\text\{\s*)?([A-D])\b',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


def run_type(ma_dang):
    spec = TYPE_PROMPTS[ma_dang]
    free_unknowns = set(spec['free_unknowns'])
    max_tool_calls = spec['max_tool_calls']
    records = ALL_RECORDS[ma_dang]
    if RUN_STT is not None:
        records = [r for r in records if r['STT'] in RUN_STT]
    out_path = f'/kaggle/working/{ma_dang.lower()}_react_calculator_full90.csv'
    print('=' * 72)
    print(f'{ma_dang}: {len(records)} questions')
    print('=' * 72)

    states = []
    for r in records:
        prompt_text = spec['template'].replace('{de_bai}', r['de_bai_mcq'])
        chat = tok.apply_chat_template([{'role': 'user', 'content': prompt_text}],
                                       tokenize=False, add_generation_prompt=True, enable_thinking=True)
        states.append(QuestionState(r, chat, free_unknowns))

    t0 = time.time()
    round_no = 0
    while True:
        active = [s for s in states if not s.done]
        if not active:
            break
        round_no += 1

        batch = []
        for s in active:
            model_input = s.chat_prompt + s.full_text
            room = MAX_MODEL_LEN - len(tok(model_input).input_ids) - 8
            n_new = min(MAX_NEW_TOKENS - s.total_tokens, room)
            if n_new <= 0:
                s.done = True
                s.stop_reason = 'out_of_context' if room <= 0 else 'out_of_token_budget'
                continue
            stop = None if s.final_round else [STOP_STR]
            batch.append((s, model_input, SamplingParams(
                temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                presence_penalty=PRESENCE_PENALTY, max_tokens=n_new,
                seed=SEED, stop=stop)))

        if not batch:
            break

        outs = llm.generate([p for _, p, _ in batch],
                            [sp_ for _, _, sp_ in batch], use_tqdm=False)

        calls_this_round = 0
        for (s, _, _), out in zip(batch, outs):
            o = out.outputs[0]
            s.full_text += o.text
            s.total_tokens += len(o.token_ids)

            if s.final_round:
                s.done = True
                s.stop_reason = 'tool_limit_reached'
                continue

            if o.stop_reason != STOP_STR:
                s.done = True
                s.stop_reason = ('model_finished' if o.finish_reason == 'stop'
                                 else 'out_of_tokens')
                continue

            s.n_calls += 1
            calls_this_round += 1
            match = None
            for mm in ACTION_INPUT_RE.finditer(o.text):
                match = mm
            expression = match.group(1).strip() if match else ''

            if not expression:
                observation = ('ERROR: no "Action Input:" line found. Write a Thought, '
                               'then "Action: Calculator", then "Action Input: <expression>".')
                s.tool_log.append(('(no Action Input)', observation))
            else:
                result, error = s.calculator.evaluate(expression)
                if error:
                    observation = (f'ERROR: {error}. Fix the syntax and try again '
                                   '(sqrt() for roots, ** for powers, log() for logarithms, '
                                   'Eq(a, b) to compare).')
                    s.tool_log.append((expression, f'ERROR: {error}'))
                else:
                    observation = result
                    s.tool_log.append((expression, result))

            s.full_text += f'{STOP_STR} {observation}\n'

            if s.n_calls >= max_tool_calls:
                s.full_text += ('\n[SYSTEM: tool-call limit reached, give your Final '
                                'Answer now, with no further Action.]\n')
                s.final_round = True

        print(f'  Round {round_no:2d}: {len(batch):2d} questions generating, '
              f'{calls_this_round:2d} tool calls, '
              f'{sum(1 for s in states if not s.done):2d} still active '
              f'({time.time()-t0:.0f}s)')

    print(f'Finished after {round_no} rounds, {(time.time()-t0)/60:.1f} minutes.')

    rows = []
    for s in states:
        chosen = extract_letter(s.full_text)
        n_err = sum(1 for _, obs in s.tool_log if str(obs).startswith('ERROR'))
        rows.append({
            'STT': s.rec['STT'],
            'ma_dang': s.rec.get('ma_dang', ma_dang),
            'dap_an_dung': s.rec['dap_an_letter'],
            'model_chon': chosen,
            'DUNG': chosen == s.rec['dap_an_letter'],
            'token': s.total_tokens,
            'tool_calls': s.n_calls,
            'syntax_errors': n_err,
            'stop_reason': s.stop_reason,
            'tool_log': ' | '.join(f'{ex} -> {obs}' for ex, obs in s.tool_log),
            'full_text': s.full_text,
        })

    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False, encoding='utf-8-sig')

    n_right = int(df['DUNG'].sum())
    n_none = int((df['model_chon'] == '').sum())
    print(f"RESULT {ma_dang}: {n_right} / {len(df)} correct ({df['DUNG'].mean()*100:.1f}%)   "
          f"wrong={len(df) - n_right - n_none}   undetermined={n_none}")
    print(f"Average tool calls per question : {df['tool_calls'].mean():.1f}   "
          f"Total syntax errors: {df['syntax_errors'].sum()} "
          f"({df['syntax_errors'].sum() / max(1, df['tool_calls'].sum()) * 100:.1f}%)   "
          f"Questions with NO tool call: {(df['tool_calls'] == 0).sum()}   "
          f"Average tokens: {df['token'].mean():.0f}")
    print(df['stop_reason'].value_counts().to_string())
    if (~df['DUNG']).any():
        print('--- WRONG OR UNDETERMINED QUESTIONS ---')
        print(df[~df['DUNG']][['STT', 'dap_an_dung', 'model_chon', 'token',
                               'tool_calls', 'syntax_errors', 'stop_reason']].to_string(index=False))
    else:
        print(f'*** ALL {len(df)} QUESTIONS CORRECT ***')
    print('Saved:', out_path)
    return df


# =====================================================================
# RUN EVERY TYPE OF THIS NOTEBOOK, ONE AFTER THE OTHER
# =====================================================================
summary_rows = []
for ma in RUN_TYPES:
    d_ = run_type(ma)
    summary_rows.append({'ma_dang': ma, 'tong': len(d_), 'dung': int(d_['DUNG'].sum()),
                         'ti_le_dung': round(float(d_['DUNG'].mean()) * 100, 1),
                         'tool_calls_tb': round(float(d_['tool_calls'].mean()), 1),
                         'syntax_errors': int(d_['syntax_errors'].sum()),
                         'token_tb': round(float(d_['token'].mean()))})
    pd.DataFrame(summary_rows).to_csv(f'/kaggle/working/summary_{GROUP}.csv', index=False, encoding='utf-8-sig')

print()
print('=' * 72)
print('SUMMARY')
print('=' * 72)
print(pd.DataFrame(summary_rows).to_string(index=False))
